## Running RAG

In [6]:
from dotenv import load_dotenv 
from openai import OpenAI

import pandas as pd

load_dotenv()
openai_client = OpenAI()

# A->Q->A' Evaluation

Now we compare the RAG answer with the original answer from the FAQ to check if the RAG pipeline is producing answers which match the ground truth. 

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

from utils.evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress


/Users/evan/Projects/llm-zoomcamp-2026-code/module_04_evaluation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer." 
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' if otherwise."
    )

In [4]:
# Prompt 
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()


# prompt template we pass to the judge for each answer. 

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [7]:
df_answers = pd.read_csv('data/rag-answers-new.csv')

answers = df_answers.to_dict(orient="records")

In [9]:
rec = answers[0]

rec

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes, you can still join the course if you found it late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

### Create the judge prompt

In [13]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

print(prompt)

Question:
Can I still join the course if I found it late?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join the course if you found it late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.


## Call the judge: 

In [14]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt, 
    AnswerEvaluation
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says late joining is allowed and correctly adds the condition that to receive a certificate, the project must be submitted while submissions are still open. No key information is missing or contradicted.', score='good')

In [21]:
print(f"reasoning: {eval_result.reasoning}")

print(f"score: {eval_result.score}")

calc_price(usage)

reasoning: The AI answer matches the ground truth: it says late joining is allowed and correctly adds the condition that to receive a certificate, the project must be submitted while submissions are still open. No key information is missing or contradicted.
score: good


{'input_cost': 0.00022425,
 'output_cost': 0.0002745,
 'total_cost': 0.0004987500000000001}

#### Wrap it in a function

In [22]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

### Test it on the same record: 

In [23]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [25]:
print(f"reasoning: {eval_result.reasoning}")

print(f"score: {eval_result.score}")

calc_price(usage)

reasoning: The AI answer preserves the original meaning exactly: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. No key information is missing or incorrect.
score: good


{'input_cost': 0.00022425,
 'output_cost': 0.00022500000000000002,
 'total_cost': 0.00044925}

## Run the Judge Over Everything

In [32]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec['question'],
        answer_orig=rec['answer_orig'],
        answer_llm=rec['answer_llm']
    )

    result = {
        'question': rec["question"],
        "document": rec["document"],
        "score": eval_result.score, 
        "reasoning": eval_result.reasoning,
    }

    return result, usage

### As always, make use of parallel processing

In [33]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool: 
    results = map_progress(pool, answers, judge_record)

100%|██████████| 515/515 [02:48<00:00,  3.05it/s]


#### Split the results: 

In [35]:
evaluations = []
usages = []

for evaluation, usage in results: 
    evaluations.append(evaluation)
    usages.append(usage)


In [36]:
df_eval = pd.DataFrame(evaluations)

calc_total_price(usages)

0.35150925000000005

## Check the Results

In [46]:
good_count = (df_eval["score"]=='good').sum()

total_count = len(df_eval) 

print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 492/515 = 95.53%


In [47]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,Is it too late to start this course now?,74eb249bbf,bad,The ground truth says it is not too late to st...
9,Can I still join the LLM Zoomcamp and turn in ...,977bf7786c,bad,The AI answer does not convey the ground truth...
33,Is the Capstone project the only mandatory thi...,9f689c185f,bad,The AI answer correctly states that the Capsto...
48,What’s the best way to keep up with the latest...,31456f4b5f,bad,The ground truth says to use the LLM Zoomcamp ...
57,"Where do you announce live sessions, and how w...",d65e05bd7a,bad,The AI answer captures the key idea that live ...


In [49]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

In [50]:
df_eval.score.value_counts(normalize=True)

score
good    0.95534
bad     0.04466
Name: proportion, dtype: float64